In [29]:
import langchain

In [30]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    
    TokenTextSplitter,
)
print("Set up Completed!")


Set up Completed!


In [31]:
## create a simple document
doc=Document(
    page_content="This is the main text content that will be embedded and searched.",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Krish Naik",
        "date_created":"2024-01-01",
        "cutom_field":"any_value"

    }
)
print("Document Structure")

print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

# Why metadata matters:
print("\n📝 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")


Document Structure
Content :This is the main text content that will be embedded and searched.
Metadata :{'source': 'example.txt', 'page': 1, 'author': 'Krish Naik', 'date_created': '2024-01-01', 'cutom_field': 'any_value'}

📝 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


In [32]:
type(doc)

langchain_core.documents.base.Document

In [33]:
## Create a simple txt file
import os
os.makedirs("data/text_files",exist_ok=True)

In [34]:
sample_texts = {
    "data/text_files/passport_info.txt": """Passport Information (পাসপোর্ট তথ্য)

Bangladesh passport একটি official travel document যা Ministry of Home Affairs দ্বারা ইস্যু করা হয়।
Applicants must submit National ID (NID), birth certificate, এবং application form.


Passport validity সাধারণত 5 years অথবা 10 years হয়ে থাকে।
For passport renewal, applicants must apply before expiry date.

সাধারণ প্রশ্ন:
- How to apply for passport?
- Passport application status check
- Required documents for passport renewal
""",

    "data/text_files/brta_info.txt": """BRTA Information (BRTA তথ্য)

Bangladesh Road Transport Authority (BRTA) driving license, vehicle registration,
এবং smart card services প্রদান করে।

Driving License Types:
1. Learner License (লার্নার লাইসেন্স)
2. Professional License
3. Non-professional License

BRTA services include:
- Online driving license application
- Smart card delivery status
- Vehicle registration verification

সাধারণ প্রশ্ন:
- Driving license renewal fee কত?
- BRTA smart card কবে আসবে?
"""
}

# create files
for filepath, content in sample_texts.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ Passport & BRTA sample files created!")


✅ Passport & BRTA sample files created!


In [35]:
from langchain_community.document_loaders import TextLoader

# Loading a single Passport text file
loader = TextLoader(
    "data/text_files/passport_info.txt",
    encoding="utf-8"
)

documents = loader.load()

print(f"📄 Loaded {len(documents)} document")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")


📄 Loaded 1 document
Content preview: Passport Information (পাসপোর্ট তথ্য)

Bangladesh passport একটি official travel document যা Ministry ...
Metadata: {'source': 'data/text_files/passport_info.txt'}


In [36]:
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader=DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files  
    loader_cls= TextLoader, ##loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True

)

documents=dir_loader.load()

print(f"📁 Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")


# 📊 Analysis
print("\n📊 DirectoryLoader Characteristics:")
print("✅ Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")

print("\n❌ Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")


100%|██████████| 2/2 [00:00<00:00, 2642.08it/s]

📁 Loaded 2 documents

Document 1:
  Source: data/text_files/brta_info.txt
  Length: 466 characters

Document 2:
  Source: data/text_files/passport_info.txt
  Length: 469 characters

📊 DirectoryLoader Characteristics:
✅ Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

❌ Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


In [37]:


from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)


In [38]:
### MEthod 1- Character Text Splitter
text=documents[0].page_content
text


'BRTA Information (BRTA তথ্য)\n\nBangladesh Road Transport Authority (BRTA) driving license, vehicle registration,\nএবং smart card services প্রদান করে।\n\nDriving License Types:\n1. Learner License (লার্নার লাইসেন্স)\n2. Professional License\n3. Non-professional License\n\nBRTA services include:\n- Online driving license application\n- Smart card delivery status\n- Vehicle registration verification\n\nসাধারণ প্রশ্ন:\n- Driving license renewal fee কত?\n- BRTA smart card কবে আসবে?\n'

In [39]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator=" ",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks=char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1️⃣ CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: BRTA Information (BRTA তথ্য)

Bangladesh Road Transport Authority (BRTA) driving license, vehicle re...


In [40]:
print(char_chunks[0])
print("------------------")
print(char_chunks[1])

BRTA Information (BRTA তথ্য)

Bangladesh Road Transport Authority (BRTA) driving license, vehicle registration,
এবং smart card services প্রদান করে।

Driving License Types:
1. Learner License (লার্নার
------------------
License (লার্নার লাইসেন্স)
2. Professional License
3. Non-professional License

BRTA services include:
- Online driving license application
- Smart card delivery status
- Vehicle registration


In [41]:
# Method 1: Character-based splitting
print("1️⃣ CHARACTER TEXT SPLITTER")
char_splitter = CharacterTextSplitter(
    separator="\n",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks=char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

1️⃣ CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: BRTA Information (BRTA তথ্য)
Bangladesh Road Transport Authority (BRTA) driving license, vehicle reg...


In [42]:
print(char_chunks[0])
print("-------------")
print(char_chunks[1])
print("-------------")
print(char_chunks[2])

BRTA Information (BRTA তথ্য)
Bangladesh Road Transport Authority (BRTA) driving license, vehicle registration,
এবং smart card services প্রদান করে।
Driving License Types:
-------------
1. Learner License (লার্নার লাইসেন্স)
2. Professional License
3. Non-professional License
BRTA services include:
- Online driving license application
- Smart card delivery status
-------------
- Vehicle registration verification
সাধারণ প্রশ্ন:
- Driving license renewal fee কত?
- BRTA smart card কবে আসবে?


In [43]:
# Method 2: Recursive character splitting (RECOMMENDED)
print("\n2️⃣ RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Try these separators in order
    chunk_size=200,
    chunk_overlap=20,
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chunks)} chunks")
print(f"First chunk: {recursive_chunks[0][:100]}...")


2️⃣ RECURSIVE CHARACTER TEXT SPLITTER
Created 3 chunks
First chunk: BRTA Information (BRTA তথ্য)

Bangladesh Road Transport Authority (BRTA) driving license, vehicle re...


In [44]:
print(recursive_chunks[0])
print("-----------------")
print(recursive_chunks[1])
print("------------------")
print(recursive_chunks[2])


BRTA Information (BRTA তথ্য)

Bangladesh Road Transport Authority (BRTA) driving license, vehicle registration,
এবং smart card services প্রদান করে।

Driving License Types:
1. Learner License (লার্নার
-----------------
License (লার্নার লাইসেন্স)
2. Professional License
3. Non-professional License

BRTA services include:
- Online driving license application
- Smart card delivery status
- Vehicle registration
------------------
registration verification

সাধারণ প্রশ্ন:
- Driving license renewal fee কত?
- BRTA smart card কবে আসবে?


In [45]:
# Create text without natural break points
simple_text = "This is sentence one and it is quite long. This is sentence two and it is also quite long. This is sentence three which is even longer than the others. This is sentence four. This is sentence five. This is sentence six."

splitter = RecursiveCharacterTextSplitter(
    separators=[" "],  # Only split on spaces
    chunk_size=80,
    chunk_overlap=20,
    length_function=len
)

chunks = splitter.split_text(simple_text)

print(f"\nSimple text example - {len(chunks)} chunks:\n")

for i in range(len(chunks) - 1):
    print(f"Chunk {i+1}: '{chunks[i]}'")
    print(f"Chunk {i+2}: '{chunks[i+1]}'")
    
    
    print()


Simple text example - 4 chunks:

Chunk 1: 'This is sentence one and it is quite long. This is sentence two and it is also'
Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'

Chunk 2: 'two and it is also quite long. This is sentence three which is even longer than'
Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'

Chunk 3: 'is even longer than the others. This is sentence four. This is sentence five.'
Chunk 4: 'is sentence five. This is sentence six.'



In [46]:
# Method 3: Token-based splitting
print("\n3️⃣ TOKEN TEXT SPLITTER")
token_splitter = TokenTextSplitter(
    chunk_size=50,  # Size in tokens (not characters)
    chunk_overlap=10
)

token_chunks = token_splitter.split_text(text)
print(f"Created {len(token_chunks)} chunks")
print(f"First chunk: {token_chunks[0][:100]}...")


3️⃣ TOKEN TEXT SPLITTER
Created 6 chunks
First chunk: BRTA Information (BRTA তথ্য)

Bangladesh Road Transport Authority (BRTA) driving license, vehicle re...


In [47]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")
print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  Use when: Working with token-limited models
